# Sun oxide GW BO value pilot

Use a **standard Colab CPU runtime** (not GPU/A100) and choose **Runtime -> Run all**. The notebook resolves GitHub `main` once, checks out that exact preregistration commit detached, provisions the proven isolated Python 3.12.13 environment, runs oracle-isolated smoke checks, and only then runs the frozen `NO_PBE` versus `FULL_PBE` pilot. Do not install anything manually.


In [ ]:
import hashlib
import importlib.metadata as md
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import zipfile

TERMINAL_STATES = {
    'PASS_PBE_VALUE_COLAB', 'FAIL_PBE_VALUE_COLAB',
    'LAPLACE_VALIDATION_BLOCKED', 'LAPLACE_VALIDATION_FAILED',
    'NUMERICAL_FAILURE_COLAB', 'INSTALLATION_BLOCKED',
}
TERMINAL_STATE = None
REPOSITORY_URL = 'https://github.com/PaulsonLab/energy-inference-bo.git'
REPOSITORY_DIR = Path('/content/energy-inference-bo')
try:
    resolved = subprocess.run(
        ['git', 'ls-remote', REPOSITORY_URL, 'refs/heads/main'],
        check=True, capture_output=True, text=True,
    ).stdout.strip().split()
    if len(resolved) != 2 or resolved[1] != 'refs/heads/main':
        raise RuntimeError('Could not resolve the preregistered main commit')
    RUN_SHA = resolved[0]
    if REPOSITORY_DIR.exists():
        shutil.rmtree(REPOSITORY_DIR)
    subprocess.run(
        ['git', 'clone', '--branch', 'main', '--single-branch', REPOSITORY_URL, str(REPOSITORY_DIR)],
        check=True,
    )
    subprocess.run(['git', 'checkout', '--detach', RUN_SHA], cwd=REPOSITORY_DIR, check=True)
    observed_sha = subprocess.run(
        ['git', 'rev-parse', 'HEAD'], cwd=REPOSITORY_DIR, check=True,
        capture_output=True, text=True,
    ).stdout.strip()
    if observed_sha != RUN_SHA:
        raise RuntimeError((observed_sha, RUN_SHA))
    print('RUN_SHA', RUN_SHA)
except Exception as exc:
    print('INSTALLATION_ERROR', type(exc).__name__, str(exc))
    TERMINAL_STATE = 'INSTALLATION_BLOCKED'


In [ ]:
if TERMINAL_STATE is None:
    VENV_DIR = Path('/content/sunoxide_bo_value_venv')
    UV_PYTHON_DIR = Path('/content/sunoxide_bo_value_uv_python')
    LOCK_PATH = REPOSITORY_DIR / 'experiments/sun_oxide/requirements-colab-graph.txt'
    CONFIG_PATH = REPOSITORY_DIR / 'experiments/sun_oxide/configs/bo_value_pilot.json'
    UV_BOOTSTRAP_VERSION = '0.10.11'
    PYTHON_VERSION = '3.12.13'
    required_versions = {
        'numpy': '2.3.5', 'pandas': '2.3.3', 'scipy': '1.18.0',
        'matplotlib': '3.11.1',
    }
    try:
        config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
        observed_lock_hash = hashlib.sha256(LOCK_PATH.read_bytes()).hexdigest()
        if observed_lock_hash != config['environment']['lock_sha256']:
            raise RuntimeError('Frozen environment lock hash mismatch')
        if VENV_DIR.exists():
            shutil.rmtree(VENV_DIR)
        if UV_PYTHON_DIR.exists():
            shutil.rmtree(UV_PYTHON_DIR)
        subprocess.run([
            sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
            f'uv=={UV_BOOTSTRAP_VERSION}',
        ], check=True)
        uv_environment = os.environ.copy()
        uv_environment['UV_PYTHON_INSTALL_DIR'] = str(UV_PYTHON_DIR)
        subprocess.run([
            sys.executable, '-m', 'uv', 'python', 'install', PYTHON_VERSION,
            '--install-dir', str(UV_PYTHON_DIR), '--no-bin',
        ], check=True, env=uv_environment)
        subprocess.run([
            sys.executable, '-m', 'uv', 'venv', '--python', PYTHON_VERSION,
            '--managed-python', str(VENV_DIR),
        ], check=True, env=uv_environment)
        VENV_PYTHON = VENV_DIR / 'bin/python'
        subprocess.run([str(VENV_PYTHON), '-m', 'ensurepip', '--upgrade'], check=True)
        subprocess.run([
            str(VENV_PYTHON), '-m', 'pip', 'install', '--require-hashes',
            '--no-deps', '-r', str(LOCK_PATH),
        ], check=True)
        subprocess.run([str(VENV_PYTHON), '-m', 'pip', 'check'], check=True)
        version_code = (
            "import importlib.metadata as md,json,sys; "
            "required=" + repr(required_versions) + "; "
            "observed={k:md.version(k) for k in required}; "
            "assert sys.version.split()[0]=='3.12.13',sys.version; "
            "assert observed==required,(observed,required); "
            "print(json.dumps({'python':sys.version.split()[0],'versions':observed},sort_keys=True))"
        )
        version_check = subprocess.run(
            [str(VENV_PYTHON), '-c', version_code], check=True,
            capture_output=True, text=True,
        )
        print(version_check.stdout.strip())
    except Exception as exc:
        print('INSTALLATION_ERROR', type(exc).__name__, str(exc))
        TERMINAL_STATE = 'INSTALLATION_BLOCKED'


In [ ]:
if TERMINAL_STATE is None:
    try:
        smoke_command = [
            str(VENV_PYTHON),
            str(REPOSITORY_DIR / 'experiments/sun_oxide/bo_value_pilot.py'),
            'smoke', '--config', str(CONFIG_PATH),
            '--repository-root', str(REPOSITORY_DIR), '--run-sha', RUN_SHA,
        ]
        smoke = subprocess.run(
            smoke_command, cwd=REPOSITORY_DIR, check=True,
            capture_output=True, text=True,
        )
        print(smoke.stdout, end='')
        if 'SMOKE_PASS' not in smoke.stdout:
            raise RuntimeError('Oracle-isolated scientific smoke did not pass')
    except Exception as exc:
        if isinstance(exc, subprocess.CalledProcessError):
            print(exc.stdout or '', end='')
            print(exc.stderr or '', end='')
        print('INSTALLATION_ERROR', type(exc).__name__, str(exc))
        TERMINAL_STATE = 'INSTALLATION_BLOCKED'


In [ ]:
if TERMINAL_STATE is None:
    OUTPUT_DIR = Path('/content/sun_oxide_bo_value_pilot_outputs')
    ZIP_PATH = Path('/content/sun_oxide_bo_value_pilot_outputs.zip')
    if OUTPUT_DIR.exists():
        shutil.rmtree(OUTPUT_DIR)
    if ZIP_PATH.exists():
        ZIP_PATH.unlink()
    run_command = [
        str(VENV_PYTHON),
        str(REPOSITORY_DIR / 'experiments/sun_oxide/bo_value_pilot.py'),
        'run', '--config', str(CONFIG_PATH),
        '--repository-root', str(REPOSITORY_DIR), '--run-sha', RUN_SHA,
        '--output-dir', str(OUTPUT_DIR), '--zip-path', str(ZIP_PATH),
    ]
    try:
        process = subprocess.Popen(
            run_command, cwd=REPOSITORY_DIR, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            stripped = line.strip()
            if stripped in TERMINAL_STATES:
                TERMINAL_STATE = stripped
            else:
                print(line, end='')
        return_code = process.wait()
        if return_code != 0 or TERMINAL_STATE is None:
            TERMINAL_STATE = 'NUMERICAL_FAILURE_COLAB'
        if ZIP_PATH.is_file():
            manifest = json.loads((OUTPUT_DIR / 'artifact_manifest.json').read_text(encoding='utf-8'))
            if manifest['run_sha'] != RUN_SHA:
                raise RuntimeError('Output RUN_SHA mismatch')
            for entry in manifest['files']:
                artifact = OUTPUT_DIR / entry['path']
                if artifact.stat().st_size != entry['size_bytes']:
                    raise RuntimeError('Artifact size mismatch: ' + entry['path'])
                if hashlib.sha256(artifact.read_bytes()).hexdigest() != entry['sha256']:
                    raise RuntimeError('Artifact hash mismatch: ' + entry['path'])
            with zipfile.ZipFile(ZIP_PATH, 'r') as archive:
                expected = [entry['path'] for entry in manifest['files']] + ['artifact_manifest.json']
                if archive.namelist() != expected:
                    raise RuntimeError('ZIP member list mismatch')
            print('VERIFIED_ZIP_SHA256', hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest())
    except Exception as exc:
        print('RUN_ERROR', type(exc).__name__, str(exc))
        TERMINAL_STATE = 'NUMERICAL_FAILURE_COLAB'


In [ ]:
if TERMINAL_STATE is None:
    TERMINAL_STATE = 'INSTALLATION_BLOCKED'
print(TERMINAL_STATE)
